# Extract the Entities of the 1914 Press

Which people, places and organizations does a passage name? Three methods, from the fastest to the most flexible, compared on 40 hand-annotated passages (in French, like the corpus):

1. spaCy, its fixed labels
2. GLiNER, labels of your choice
3. An LLM filling in a schema
4. Link names to a reference list
5. Compare, and spot inventions

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and Tools

The LLM runs in [Ollama](https://ollama.com), on the Colab machine: the next cell installs it and downloads the model in the background, while you work on spaCy and GLiNER.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q gliner rapidfuzz
!python -m spacy download fr_core_news_md > /dev/null

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import json

import pandas as pd
from rapidfuzz import fuzz

gold = pd.read_json("gold.jsonl", lines=True)
FIELDS = ["people", "places", "organizations"]
print(len(gold), "annotated passages")
gold.head(3)

`evaluate(predictions)` compares extractions (a list of dictionaries `{"people": [...], "places": [...], "organizations": [...]}`, in `gold`'s order) with the annotations: for each field, the precision (the share of extracted entities that are right), the recall (the share of expected entities that are found) and their F1. Matching is lenient: two names match if their Levenshtein similarity (`fuzz.ratio`) is at least 85 out of 100, or if one contains the other ("Jaurès" and "Jean Jaurès").

In [ ]:
def normalize(name: str) -> str:
    return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
    a, b = normalize(a), normalize(b)
    return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.DataFrame:
    rows = {}
    for field in FIELDS:
        found = expected = right_found = right_expected = 0
        for prediction, (_, row) in zip(predictions, gold.iterrows()):
            predicted = list(dict.fromkeys(prediction.get(field, [])))
            truth = row[field]
            found += len(predicted)
            expected += len(truth)
            right_found += sum(any(same(p, t) for t in truth) for p in predicted)
            right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
        precision = right_found / max(found, 1)
        recall = right_expected / max(expected, 1)
        rows[field] = {"precision": precision, "recall": recall, "F1": 2 * precision * recall / max(precision + recall, 1e-9)}
    return pd.DataFrame(rows).T.round(2)


results = {}

## 1. spaCy

Load `fr_core_news_md` (the passages are French) and extract each passage's entities: `PER` for people, `LOC` for places, `ORG` for organizations. Evaluate, and store the table in `results["spaCy"]`. How long for the 40 passages?

In [ ]:
# Your code here

### Solution

In [ ]:
import time

import spacy

nlp = spacy.load("fr_core_news_md")
LABELS = {"PER": "people", "LOC": "places", "ORG": "organizations"}
start = time.time()
spacy_predictions = []
for doc in nlp.pipe(gold.text):
    prediction = {field: [] for field in FIELDS}
    for ent in doc.ents:
        if ent.label_ in LABELS:
            prediction[LABELS[ent.label_]].append(ent.text)
    spacy_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["spaCy"] = evaluate(spacy_predictions)
results["spaCy"]

## 2. GLiNER

[GLiNER](https://github.com/urchade/GLiNER) finds the entities of the labels you give it, without training. Load `urchade/gliner_multi-v2.1` (on the GPU, `.to("cuda")`) and extract the entities of the labels `["person", "place", "organization"]` (`model.predict_entities(text, labels, threshold=0.4)`). Evaluate. Try other wordings of the labels ("newspaper", "political party"…), or another threshold.

In [ ]:
# Your code here

### Solution

In [ ]:
from gliner import GLiNER

gliner = GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to("cuda")
FIELD_OF = {"person": "people", "place": "places", "organization": "organizations"}
start = time.time()
gliner_predictions = []
for text in gold.text:
    prediction = {field: [] for field in FIELDS}
    for entity in gliner.predict_entities(text, list(FIELD_OF), threshold=0.4):
        prediction[FIELD_OF[entity["label"]]].append(entity["text"])
    gliner_predictions.append(prediction)
print(f"{time.time() - start:.1f} s")
results["GLiNER"] = evaluate(gliner_predictions)
results["GLiNER"]

## 3. An LLM Filling in a Schema

The model runs in Ollama, which offers an OpenAI-compatible API: the same code would work with a commercial API, changing the address and the key.

Describe the expected result with a Pydantic class `Entities` (three lists of strings), then write `extract(text)` that asks the model for the passage's entities, with `client.chat.completions.parse(model=MODEL, messages=..., response_format=Entities, temperature=0)`: the answer necessarily follows the schema. Your prompt must give the annotation guidelines (below). Run the extraction of the 40 passages **in parallel** (`ThreadPoolExecutor(8)`), time it, evaluate.

Annotation guidelines: each entity named once, as written in the text (OCR errors included); people without their title (M., Mme, sir); places (countries, towns, streets, buildings); named organizations (assemblies, parties, unions, newspapers, agencies); no generic mentions ("le gouvernement").

In [ ]:
pull.wait()  # the model is downloaded

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# A first call loads the model onto the GPU, where it stays: before any parallel work.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1)
print(MODEL, "ready")

In [ ]:
# Your code here

### Solution

In [ ]:
from concurrent.futures import ThreadPoolExecutor


class Entities(BaseModel):
    people: list[str]
    places: list[str]
    organizations: list[str]


INSTRUCTIONS = """Extract the named entities of this passage of the 1914 French press.
- people: persons, without their title (M., Mme, sir, lord);
- places: places (countries, regions, towns, streets, buildings);
- organizations: named organizations (assemblies, parties, unions, newspapers, agencies).
Each entity once, as written in the text, errors included.
No generic mentions such as "le gouvernement". Do not invent anything."""


def extract(text: str) -> dict[str, list[str]]:
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": text}],
        response_format=Entities,
        temperature=0,
    )
    return response.choices[0].message.parsed.model_dump()


start = time.time()
with ThreadPoolExecutor(8) as pool:
    llm_predictions = list(pool.map(extract, gold.text))
print(f"{time.time() - start:.1f} s")
results["LLM"] = evaluate(llm_predictions)
results["LLM"]

## 4. Link Names to a Reference List

The same people appear in different forms ("Jaurès", "Jean Jaurès", "M. Jaurès"), sometimes misread by the OCR. For each person the LLM extracted, look up the closest name in `REFERENCE` (`rapidfuzz.process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)`), and keep it if the score is at least 85. Which people of the corpus get linked?

In [ ]:
REFERENCE = [
    "Jean Jaurès", "Joseph Caillaux", "Henriette Caillaux", "Gaston Calmette", "Raymond Poincaré",
    "René Viviani", "Georges Clemenceau", "Jules Guesde", "Léon Jouhaux", "Édouard Vaillant",
    "Edward Grey", "Herbert Asquith", "Paul Cambon", "Raoul Villain", "Gabriel Hanotaux",
    "Antonino di San Giuliano", "Émile Vandervelde", "John Simon", "Woodrow Wilson",
]

In [ ]:
# Your code here

### Solution

In [ ]:
from rapidfuzz import process

for name in sorted({name for prediction in llm_predictions for name in prediction["people"]}):
    match, score, _ = process.extractOne(name, REFERENCE, scorer=fuzz.WRatio)
    if score >= 85:
        print(f"{name:25} -> {match} ({score:.0f})")

## 5. Compare, and Spot Inventions

Gather the three methods' F1 in one table. Then look for the LLM's **inventions**: extracted entities that don't appear in the passage's text (`fuzz.partial_ratio(normalize(entity), normalize(text)) < 85`). Are there any? Which of the three methods would you choose for 10 million passages?

In [ ]:
# Your code here

### Solution

In [ ]:
print(pd.DataFrame({name: table["F1"] for name, table in results.items()}))

for prediction, text in zip(llm_predictions, gold.text):
    for field in FIELDS:
        for entity in prediction[field]:
            if fuzz.partial_ratio(normalize(entity), normalize(text)) < 85:
                print(f"{field:13} {entity!r} is not in the text")

# spaCy is by far the fastest, but its labels are fixed and it errs on noisy text; GLiNER
# is a good compromise; the LLM is the most flexible and usually the most accurate, but the
# slowest and the most expensive: for 10 million passages, we'd use it to label examples,
# then fine-tune a small model (next lab).